In [1]:
from pathlib import Path

text = Path("./../data/tiny-shakespeare.txt").read_text()

In [2]:
print(text[0:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [3]:
import torch


class CharTokenizer:
    def __init__(self, vocabulary):
        self.token_id_for_char = {
            char: token_id for token_id, char in enumerate(vocabulary)
        }
        self.char_for_token_id = {
            token_id: char for token_id, char in enumerate(vocabulary)
        }

    @staticmethod
    def train_from_text(text):
        vocabulary = set(text)
        return CharTokenizer(sorted(list(vocabulary)))

    def encode(self, text):
        token_ids = []
        for char in text:
            token_ids.append(self.token_id_for_char[char])
        return torch.tensor(token_ids, dtype=torch.long)

    def decode(self, token_ids):
        chars = []
        for token_id in token_ids.tolist():
            chars.append(self.char_for_token_id[token_id])
        return "".join(chars)

    def vocabulary_size(self):
        return len(self.token_id_for_char)

In [4]:
tokenizer = CharTokenizer.train_from_text(text)

In [5]:
print(tokenizer.encode("Hello world"))
print(tokenizer.decode(tokenizer.encode("Hello world")))

tensor([20, 43, 50, 50, 53,  1, 61, 53, 56, 50, 42])
Hello world


In [6]:
print(f"Vocabulary size: {tokenizer.vocabulary_size()}")

Vocabulary size: 65


In [7]:
# Step 1 - Define the `TokenIdsDataset` Class

from torch.utils.data import Dataset

class TokenIdsDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, pos):
        assert pos < len(self.data) - self.block_size
        
        pos_item = self.data[pos:pos + self.block_size]
        target_item = self.data[pos+1:pos + 1 + self.block_size]

        return pos_item, target_item


In [8]:
# Step 2 - Tokenize the Text

# TODO: Encode text using the tokenizer
tokenized = tokenizer.encode(text)
# Create "TokenIdsDataset" with the tokenized text, and block_size=64
token_dataset = TokenIdsDataset(tokenizer.encode(text),block_size=64)

In [9]:
# Step 3 - Retrieve the First Item from the Dataset

# TODO: Get the first item from the dataset
print(token_dataset[0])

pos, target = token_dataset[0]
print(f"pos:{pos}")
print(tokenizer.decode(pos))
# Decode "x" using tokenizer.decode
print("="*10)
print(tokenizer.decode(target))
print(f"target:{target}")

(tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50]), tensor([47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44, 53,
        56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,  1,
        44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1, 57,
        54, 43, 39, 49,  8,  0,  0, 13, 50, 50]))
pos:tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50])
First Citizen:
Before we proceed any further, hear me speak.

Al
irst Citizen:
Before we proceed any further, hear me speak.



In [10]:
from torch.utils.data import DataLoader, RandomSampler

# RandomSampler allows to read random items from a datasset
sampler = RandomSampler(token_dataset, replacement=True)
# Dataloader will laod two random samplers using the sampler
dataloader = DataLoader(token_dataset, batch_size=2, sampler=sampler)

In [11]:
# Step 4 - Use a DataLoader

# TODO: Get a single batch from the "dataloader"
dataloader_inter = iter(dataloader)
    
# For this call the `iter` function, and pass DataLoader instance to it. This will create an iterator
# Then call the `next` function and pass the iterator to it to get the first training batch
x, y = next(dataloader_inter)


In [12]:
# TODO: Decode input item
print(tokenizer.decode(x[0]))

en,
Spurn at his edict and fulfil a man's?
Take heed; for he hol


In [13]:
# TODO: Decode target item
print(tokenizer.decode(y[0]))

n,
Spurn at his edict and fulfil a man's?
Take heed; for he hold
